# Forelesning 9: Fartslover og koblede reaksjoner

I dag bygger vi videre på Eulers metode og simulerer

- reaksjoner med flere stoffer: A + B → C
- reversible reaksjoner og dynamisk likevekt: A ⇌ B
- konsekutive reaksjoner med et mellomprodukt: A → B → C

Til slutt ser vi kort på bedre metoder og ferdige løsere.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 14, "figure.figsize": (8, 4.5)})

<div class="oppgave">

**Finn feilen (repetisjon):** Programmet skal simulere en førsteordens reaksjon med $k = 0{,}10\ \mathrm{s^{-1}}$, men det inneholder **to** feil. Kjør koden, les feilmeldingen og rett feilene. Forklar for sidemannen hva hver feil gjorde.

</div>

In [ ]:
k = 0.10
A = 1.00
t = 0
dt = 0.5

times = [t]
conc = [A]

while t < 30:
    dA_dt = -k * A          # Feil 1: A forbrukes, så fortegnet må være negativt
    A = A + dA_dt * dt
    t = t + dt
    times.append(t)
    conc.append(A)          # Feil 2: må stå inne i løkka

plt.plot(times, conc)
plt.xlabel("Time (s)")
plt.ylabel("[A] (mol/L)")
plt.show()

<div class="viktig">

Fortegnet i en fartslov forteller om stoffet **forbrukes** (negativt) eller **dannes** (positivt). Får du en konsentrasjon som vokser når den burde avta, sjekk fortegnet først.

</div>

## Del 1: A + B → C

Farten er $r = k[\mathrm{A}][\mathrm{B}]$. Støkiometrien bestemmer fortegnene:

<div class="viktig">

$$\frac{d[\mathrm{A}]}{dt} = -r \qquad \frac{d[\mathrm{B}]}{dt} = -r \qquad \frac{d[\mathrm{C}]}{dt} = +r$$

Likningene er **koblet**: Den samme farten avhenger av både $[\mathrm{A}]$ og $[\mathrm{B}]$, så alle konsentrasjonene må oppdateres i samme tidssteg.

</div>

Nå lagrer vi konsentrasjonene i arrayer. `A[n]` er konsentrasjonen i tidssteg nummer `n`.

<div class="oppgave">

**Prøv selv:** Fyll inn Euler-stegene for B og C. Bruk $k = 0{,}08$ L/(mol·s), $[\mathrm{A}]_0 = 1{,}00$ M og $[\mathrm{B}]_0 = 0{,}80$ M.

</div>

In [ ]:
k = 0.08
dt = 0.1
t = np.arange(0, 60 + dt, dt)

A = np.zeros(len(t))
B = np.zeros(len(t))
C = np.zeros(len(t))
A[0] = 1.00
B[0] = 0.80

for n in range(len(t) - 1):
    rate = k * A[n] * B[n]
    A[n + 1] = A[n] - rate * dt
    B[n + 1] = B[n] - rate * dt
    C[n + 1] = C[n] + rate * dt

plt.plot(t, A, label="[A]")
plt.plot(t, B, label="[B]")
plt.plot(t, C, label="[C]")
plt.xlabel("Time (s)")
plt.ylabel("Concentration (mol/L)")
plt.legend()
plt.show()

<div class="diskuter">

**Diskuter:** Hvorfor går ikke [A] mot null? Hvilken konsentrasjon nærmer [C] seg, og hvorfor?

</div>

### Kontroller kjemien: stoffbalanse
Et program er ikke riktig bare fordi grafen ser pen ut. Med bare denne reaksjonen skal $[\mathrm{A}] + [\mathrm{C}] = [\mathrm{A}]_0$ og $[\mathrm{B}] + [\mathrm{C}] = [\mathrm{B}]_0$ hele tiden.

**Fyll inn** kontrollen for B.

In [ ]:
print("Largest deviation, A-balance:", np.max(np.abs(A + C - A[0])))
print("Largest deviation, B-balance:", np.max(np.abs(B + C - B[0])))

## Del 2: Reversibel reaksjon A ⇌ B

<div class="viktig">

$$\frac{d[\mathrm{A}]}{dt} = -k_f[\mathrm{A}] + k_b[\mathrm{B}] \qquad \frac{d[\mathrm{B}]}{dt} = k_f[\mathrm{A}] - k_b[\mathrm{B}]$$

Ved **dynamisk likevekt** er nettofarten null, men både framover- og bakoverreaksjonen går fortsatt.

</div>

<div class="oppgave">

**Prøv selv:** Fyll inn nettofarten $k_f[\mathrm{A}] - k_b[\mathrm{B}]$ og Euler-stegene. Start med bare A. Sammenlikn $[\mathrm{B}]/[\mathrm{A}]$ ved slutten med $k_f/k_b$.

</div>

In [ ]:
k_f = 0.30
k_b = 0.10
dt = 0.05
t = np.arange(0, 30 + dt, dt)

A = np.zeros(len(t))
B = np.zeros(len(t))
A[0] = 1.0

for n in range(len(t) - 1):
    net = k_f * A[n] - k_b * B[n]
    A[n + 1] = A[n] - net * dt
    B[n + 1] = B[n] + net * dt

plt.plot(t, A, label="[A]")
plt.plot(t, B, label="[B]")
plt.xlabel("Time (s)")
plt.ylabel("Concentration (mol/L)")
plt.legend()
plt.show()

print("B/A at the end:", B[-1] / A[-1])
print("k_f/k_b:", k_f / k_b)

<div class="viktig">

Ved likevekt er $k_f[\mathrm{A}] = k_b[\mathrm{B}]$, altså $\dfrac{[\mathrm{B}]}{[\mathrm{A}]} = \dfrac{k_f}{k_b} = K$. Simuleringen gir likevektskonstanten fra kinetikken.

</div>

<div class="diskuter">

**Diskuter:** Hva skjer hvis vi starter med bare B? Ender vi på den samme likevekten? Tipp først, og test.

</div>

## Del 3: Konsekutive reaksjoner A → B → C

Mange reaksjoner går via et **mellomprodukt**. Med $k_1 = 0{,}50\ \mathrm{s^{-1}}$ og $k_2 = 0{,}10\ \mathrm{s^{-1}}$:

<div class="viktig">

$$\frac{d[\mathrm{A}]}{dt} = -k_1[\mathrm{A}] \qquad \frac{d[\mathrm{B}]}{dt} = k_1[\mathrm{A}] - k_2[\mathrm{B}] \qquad \frac{d[\mathrm{C}]}{dt} = k_2[\mathrm{B}]$$

</div>

<div class="oppgave">

**Prøv selv:** Skriv de tre Euler-stegene selv. Finn tidspunktet der [B] er størst med `np.argmax`.

</div>

In [ ]:
k1 = 0.50
k2 = 0.10
dt = 0.01
t = np.arange(0, 40 + dt, dt)

A = np.zeros(len(t))
B = np.zeros(len(t))
C = np.zeros(len(t))
A[0] = 1.0

for n in range(len(t) - 1):
    A[n + 1] = A[n] - k1 * A[n] * dt
    B[n + 1] = B[n] + (k1 * A[n] - k2 * B[n]) * dt
    C[n + 1] = C[n] + k2 * B[n] * dt

plt.plot(t, A, label="[A]")
plt.plot(t, B, label="[B]")
plt.plot(t, C, label="[C]")
plt.xlabel("Time (s)")
plt.ylabel("Concentration (mol/L)")
plt.legend()
plt.show()

i_max = np.argmax(B)
print(f"[B] is largest at t = {t[i_max]:.2f} s")

<div class="diskuter">

**Diskuter:** Hvorfor øker [B] først og avtar deretter? Hva er $d[\mathrm{B}]/dt$ i maksimumspunktet? Hva skjer med kurvene hvis $k_2$ er mye større enn $k_1$?

</div>

<div class="viktig">

I maksimum er $d[\mathrm{B}]/dt = 0$, altså dannes og forbrukes B like raskt. Analytisk ligger maksimum ved $t_{\max} = \dfrac{\ln(k_1/k_2)}{k_1 - k_2} = 4{,}02$ s. Det er en god kontroll av simuleringen.

</div>

## Del 4: Bedre metoder og ferdige løsere

<div class="bilde-hoyre" style="width:38%">

![Runge-Kutta 4 og Eulers metode](bilder/rk4_euler_meme.png)

</div>

Euler bruker bare stigningen i **starten** av hvert steg. **Runge-Kutta 4** (RK4) bruker fire stigninger i hvert steg og veier dem sammen.

<div class="viktig">

| Metode | Feilen når $\Delta t$ halveres |
|---|---|
| Euler (orden 1) | halveres |
| RK4 (orden 4) | blir 16 ganger mindre |

For $d[\mathrm{A}]/dt = -0{,}15[\mathrm{A}]$ med $\Delta t = 1$ s er feilen etter 10 s omtrent $3 \cdot 10^{-2}$ med Euler og $2 \cdot 10^{-6}$ med RK4.

</div>

### `solve_ivp`
I praksis bruker vi testede løsere. `solve_ivp` fra SciPy velger selv tidssteget. Vi skriver modellen som én funksjon som tar inn alle konsentrasjonene og returnerer alle de deriverte.

In [ ]:
from scipy.integrate import solve_ivp

def consecutive(t, y):
    A, B, C = y
    return [-k1 * A, k1 * A - k2 * B, k2 * B]

solution = solve_ivp(consecutive, [0, 40], [1.0, 0.0, 0.0], t_eval=np.linspace(0, 40, 401))

for i, name in enumerate(["[A]", "[B]", "[C]"]):
    plt.plot(solution.t, solution.y[i], label=name)
plt.xlabel("Time (s)")
plt.ylabel("Concentration (mol/L)")
plt.legend()
plt.show()

<div class="viktig">

**En god simuleringsvane**

1. Er fortegnene i fartslovene riktige ut fra støkiometrien?
2. Endrer løsningen seg hvis tidssteget reduseres?
3. Er stoffbalansen bevart?
4. Stemmer grensetilfellene med kjemisk intuisjon, for eksempel likevekt eller fullstendig omsetning?

</div>

__Oppgaver__:
1. Simuler A + B → C med $k = 0{,}05$ L/(mol·s), $[\mathrm{A}]_0 = 1{,}00$ M og $[\mathrm{B}]_0 = 0{,}60$ M. Kontroller stoffbalansen. Hva skjer når de to startkonsentrasjonene er like?
2. For A ⇌ B med $k_f = 0{,}40\ \mathrm{s^{-1}}$ og $k_b = 0{,}10\ \mathrm{s^{-1}}$: Simuler til systemet er nær likevekt. Finn omtrent når $[\mathrm{B}]$ har nådd 99 % av likevektsverdien.
3. For A → B → C: Hvordan endrer tidspunktet og høyden til maksimum for [B] seg når $k_2$ øker? Test minst fire verdier og lag et plott.
4. Løs førsteordensreaksjonen med Euler og RK4 med samme tidssteg, og sammenlikn begge med den analytiske løsningen. Hvor lite må tidssteget være for at Euler skal bli like nøyaktig som RK4 med $\Delta t = 1$ s?
5. Løs A + B → C med `solve_ivp`. Undersøk feltene `success`, `message` og `nfev` i resultatet. Hva forteller de?